In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch09-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "09-04-nearest-neighbors.py"

# 新分子的鄰居投了哪些票？
使用 Maom Lab 的 MutagenLou2023 鏡像（CC BY 4.0）。第一次執行需連網取得 train_test 兩份 CSV。分析程式在獨立的 Python 程序執行。練習程式可接在寫入暫存檔的儲存格末尾，再執行該格及最後一格。

In [ ]:
%%writefile "{script}"
# MutagenLou2023 (Maom Lab mirror, CC BY 4.0; Lou et al., 2023).
# Original rows/labels preserved in the cached files. Only structure enters X.
from pathlib import Path
from urllib.request import urlopen
import csv
import hashlib
import io
import os
import tempfile
import numpy as np
from rdkit import Chem
from sklearn.model_selection import train_test_split

def mutagen_splits():
    revision = "a9780a60774eb522b78bd92b83c30d8deae00ff5"
    source = f"https://huggingface.co/datasets/maomlab/MutagenLou2023/resolve/{revision}/train_test"
    base = Path(os.environ.get("DEEPCHEM_DATA_DIR", tempfile.gettempdir())) / "mutagenlou-a9780a6"
    base.mkdir(parents=True, exist_ok=True)
    expected = {"train": "ad0d4e2850afdc07c130c3679cc8ae2ad9f7c80999158325d8de986c39f950e1",
                "test": "08786199fbbdb6db614d5f65a5abdfcd26d882f0cc844bbeb4ccfa241127b394"}
    groups = {}
    for name, digest in expected.items():
        file = base / (name + ".csv")
        raw = file.read_bytes() if file.exists() else urlopen(f"{source}/{name}.csv", timeout=90).read()
        if hashlib.sha256(raw).hexdigest() != digest:
            raise ValueError(f"MutagenLou2023 checksum mismatch: {name}")
        if not file.exists():
            file.write_bytes(raw)
        labels = {}
        for row in csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))):
            mol = Chem.MolFromSmiles(row["SMILES"])
            if mol is not None:
                y = int(row["Y"])
                assert y in (0, 1)
                labels.setdefault(Chem.MolToSmiles(mol), set()).add(y)
        groups[name] = labels
    # Preserve mirrored test membership: no test structure may enter development.
    dev = [(s, next(iter(v))) for s,v in groups["train"].items() if len(v)==1 and s not in groups["test"]]
    test = [(s, next(iter(v))) for s,v in groups["test"].items() if len(v)==1]
    fit, valid = train_test_split(dev, test_size=0.15, random_state=42, stratify=[y for _,y in dev])
    return [(np.array([s for s,_ in rows]), np.array([y for _,y in rows])) for rows in (fit,valid,test)]

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
(train, y_train), (valid, y_valid), (test, y_test) = mutagen_splits()
print("split sizes:", len(train), len(valid), len(test))
from rdkit.Chem import rdFingerprintGenerator
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
from rdkit import DataStructs
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score

def fingerprints(split):
    return [generator.GetFingerprint(Chem.MolFromSmiles(s)) for s in split]

train_fps = fingerprints(train)
def distances(fps):
    matrix = np.empty((len(fps), len(train_fps)))
    for i, fp in enumerate(fps):
        matrix[i] = 1 - np.asarray(DataStructs.BulkTanimotoSimilarity(fp, train_fps))
    return matrix

D_train = distances(train_fps)
D_valid = distances(fingerprints(valid))
D_test = distances(fingerprints(test))
choices = [1, 3, 5, 7, 9]
scores = []
for k in choices:
    model = KNeighborsClassifier(n_neighbors=k, metric="precomputed", weights="uniform")
    model.fit(D_train, y_train)
    pred = model.predict(D_valid)
    scores.append(balanced_accuracy_score(y_valid, pred))
    print("k, validation balanced accuracy:", k, scores[-1])
k = choices[np.argmax(scores)]
model = KNeighborsClassifier(n_neighbors=k, metric="precomputed").fit(D_train, y_train)
print("k, test balanced accuracy:", k, balanced_accuracy_score(y_test, model.predict(D_test)))
distance, index = model.kneighbors(D_test[:1])
votes = y_train[index[0]].astype(int)
print("query:", test[0], "similarities:", 1 - distance[0], "votes:", votes,
      "prediction, observed:", model.predict(D_test[:1])[0], y_test[0])
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.7), layout="constrained")
axes[0].plot(choices, scores, "o-", color="#286780")
axes[0].set(xlabel="Number of neighbors", ylabel="Validation balanced accuracy", xticks=choices, ylim=(0.75, 0.80))
axes[1].bar(np.arange(1, k + 1), 1 - distance[0], color=["#be783c" if v else "#286780" for v in votes])
for i, (d, vote) in enumerate(zip(distance[0], votes), 1):
    axes[1].text(i, 1 - d + 0.02, str(vote), ha="center")
axes[1].set(xlabel="Training neighbor rank (labels above bars)", ylabel="Tanimoto similarity",
            xticks=np.arange(1, k + 1), ylim=(0, 1.10))
fig.savefig(figure_dir / "09-04-nearest-neighbors.png", dpi=150)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "09-04-nearest-neighbors.png")))